# PACT：可学习 clip 上界（Parameterized Clipping Activation）

配套文章：

- 《大模型量化算法（18）：LSQ / PACT / DSQ——可学习的 scale 与 clip》
  https://lrypcy.github.io/2026/08/29/llm-quant-18-lsq-pact-dsq/ （§4 全部）
- 《大模型量化算法（11）：伪量化算子插入》 https://lrypcy.github.io/2026/08/26/llm-quant-11-fake-quant-insertion/

**一句话**：PACT = ReLU + 一个可学习的饱和上界 `α`（即 `clip(x, 0, α)` 的 `α` 变成参数）。
`ReLU = PACT with α = ∞`。量化网格变成 `{0, α/M, 2α/M, …, α}`，共 `2^b` 个电平。

## 本 notebook 的五个实验

| # | 实验 | 对应文章 | 要验证的一句话 |
|---|---|---|---|
| A | 选对 α 的增益 | 18 篇 §4.2 | 4-bit 下"选对 α"比 `α=max(x)` 高约 **5.5 dB**（8-bit 几乎无差别） |
| B | `∂ŷ/∂α` 三段式 | 18 篇 §4.2 | 范围内 `≤ 1/(2M)`、截断区恒为 `1`；极少数被截断元素承担不成比例的梯度质量 |
| C1 | 损失对 α 的高原 | 18 篇 §4.3 | 固定 α 只训权重，α 在 12–40 上几乎平坦（< 0.1 dB） |
| C2 | α 的慢动力学 | 18 篇 §4.3 | α 动力学比权重慢 1–2 个数量级；最终 α 强烈依赖初值/α 自己的 lr |
| C3 | L2 正则极难标定 | 18 篇 §4.3 | λ 稍大就把 α 直接掐死 |

> **这是合成探针任务，不是真实模型精度。** 所有数字都在下方的 teacher–student MLP / 重尾激活上成立，只验证机制方向。

## 运行方式

```bash
cd experiments/quantization/pact_learnable_clip
jupyter nbconvert --to notebook --execute --inplace pact_learnable_clip.ipynb
```

纯 numpy + matplotlib（禁止 pip install）。**随机种子固定 `SEED=0`，结果可复现。**
顶部 `MODE` 开关：`"smoke"` 为快速冒烟（默认，< 3 分钟），`"full"` 为全量（更大网格/更多步数）。


## 0. 环境与全局配置

`MODE` 决定规模。所有超参集中在 `CFG` 里，full 模式只是把网格/步数/arm 数放大。


In [1]:

import os
import json
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
MODE = "smoke"          # "smoke" | "full"

CFG = {
    # 合成数据规模 + 训练预算，全部由 MODE 控制
    "smoke": dict(
        bits=(2, 4, 8),
        snr_grid=400,            # 实验 A 的 alpha 扫描点数
        act_n=60000,             # 重尾正激活样本数（与文章 §4.2 同规模，复现 ~5.5 dB）
        act_seed=1,
        # 实验 C（teacher-student MLP）
        mlp_in=32, mlp_hidden=24, mlp_out=8, mlp_N=384, mlp_seed=7,
        plateau_alphas=(4, 8, 12, 18, 25, 40), plateau_steps=500,
        dyn_a0=(2.0, 15.0, 46.09), dyn_steps=700, dyn_alr_mult=10.0,
        l2_lambdas=(0.0, 1e-3, 1e-2), l2_steps=900, l2_alr_mult=10.0,
        main_lr=2e-3,
    ),
    "full": dict(
        bits=(2, 3, 4, 6, 8),
        snr_grid=800,
        act_n=120000,
        act_seed=1,
        mlp_in=40, mlp_hidden=32, mlp_out=8, mlp_N=512, mlp_seed=7,
        plateau_alphas=(4, 8, 12, 18, 25, 40, 60), plateau_steps=4000,
        dyn_a0=(2.0, 5.0, 15.0, 30.0, 46.09), dyn_steps=4000, dyn_alr_mult=10.0,
        l2_lambdas=(0.0, 1e-4, 1e-3, 1e-2), l2_steps=4000, l2_alr_mult=10.0,
        main_lr=2e-3,
    ),
}[MODE]

HERE = os.getcwd()
RES = os.path.join(HERE, "results")
os.makedirs(RES, exist_ok=True)

_LINES = []
def log(msg=""):
    """打印并缓存，末尾统一写入 results/stdout.txt。"""
    print(msg)
    _LINES.append(str(msg))

def savefig(fig, name):
    p = os.path.join(RES, name)
    fig.savefig(p, dpi=130, bbox_inches="tight")
    plt.close(fig)
    log(f"[save] {p}")
    return p

log(f"MODE={MODE}  CFG={CFG}")
log(f"numpy={np.__version__}  matplotlib={matplotlib.__version__}")


MODE=smoke  CFG={'bits': (2, 4, 8), 'snr_grid': 400, 'act_n': 60000, 'act_seed': 1, 'mlp_in': 32, 'mlp_hidden': 24, 'mlp_out': 8, 'mlp_N': 384, 'mlp_seed': 7, 'plateau_alphas': (4, 8, 12, 18, 25, 40), 'plateau_steps': 500, 'dyn_a0': (2.0, 15.0, 46.09), 'dyn_steps': 700, 'dyn_alr_mult': 10.0, 'l2_lambdas': (0.0, 0.001, 0.01), 'l2_steps': 900, 'l2_alr_mult': 10.0, 'main_lr': 0.002}
numpy=2.1.1  matplotlib=3.11.1


## 1. 任务与量化器

PACT 作用在 ReLU 之后的激活上：先 `y = clip(x, 0, α)`，再均匀量化到 `b` 位
（网格 `{0, α/M, …, α}`，`M = 2^b - 1`）。对 `α` 求导时 round 沿用 STE（导数为 1），
于是 `∂ŷ/∂α` 只有三段（18 篇 Eq.4）：

```
       0,                          x < 0
∂ŷ/∂α = (1/M)(⌊Mx/α⌉ - Mx/α),      0 ≤ x < α     # 范围内：舍入残差 / M, ≤ 1/(2M)
       1,                         x ≥ α         # 被截掉的部分：恒为 1
```

下面手写 PACT 前向、三段式 `α` 梯度、以及 STE 输入梯度，全部纯 numpy。


In [2]:

QMIN = lambda b: -(2 ** (b - 1))
QMAX = lambda b: 2 ** (b - 1) - 1
M_of = lambda b: 2 ** b - 1

def pact_quant(x, alpha, b):
    """PACT 前向：clip(x, 0, alpha) 后再均匀量化到 b 位。"""
    M = M_of(b)
    y = np.clip(x, 0.0, alpha)
    return np.round(y * M / alpha) * (alpha / M)

def pact_dalpha(x, alpha, b):
    """∂ŷ/∂α（18 篇 Eq.4）：范围内 = 舍入残差/M，截断区恒为 1，负区为 0。"""
    M = M_of(b)
    d = np.zeros_like(x, dtype=float)
    inr = (x >= 0) & (x < alpha)
    d[inr] = (np.round(x[inr] * M / alpha) - x[inr] * M / alpha) / M
    d[x >= alpha] = 1.0
    return d

def pact_dx(x, alpha, b):
    """输入方向：经典 STE（仅在范围内透传梯度）。"""
    return ((x >= 0) & (x < alpha)).astype(float)

def snr(x, q):
    """激活重建 SNR（能量定义）：10 log10(E[x^2] / E[(x-q)^2])。"""
    mse = np.mean((x - q) ** 2)
    if mse <= 0:
        return float("inf")
    return 10.0 * np.log10(np.mean(x ** 2) / mse)

def make_activation(n, seed):
    """重尾正激活样本：Student-t(2.5)*0.6 再 clip 到 ≥ 0。
    接近真实 LLM 激活的重尾分布（少数极端大值与大量小值）。"""
    rng = np.random.default_rng(seed)
    x = np.maximum(rng.standard_t(2.5, n) * 0.6, 0.0)
    return x

# teacher-student MLP 用的 Adam（手写，纯 numpy）
def adam_step(p, g, m, v, t, lr, b1=0.9, b2=0.999, eps=1e-8):
    m[:] = b1 * m + (1 - b1) * g
    v[:] = b2 * v + (1 - b2) * (g * g)
    mh = m / (1 - b1 ** t)
    vh = v / (1 - b2 ** t)
    p -= lr * mh / (np.sqrt(vh) + eps)
    return p


## 实验 A：4-bit 下"选对 α"比 `α=max(x)` 高约 5.5 dB

用 60000 个重尾正激活样本，对每个位宽网格搜索使重建 SNR 最大的 `α*`，
再与"不削尾、直接取 `α=max(x)`"（即 min-max 量化）的 SNR 对比。
预期：4-bit 差距约 5.5 dB，2-bit 更大，8-bit 几乎无差别（网格已足够细）。


In [3]:

bits = CFG["bits"]
x = make_activation(CFG["act_n"], CFG["act_seed"])
amax = float(x.max())
log(f"激活样本: n={x.size} mean={x.mean():.4f} rms={np.sqrt((x**2).mean()):.4f} max={amax:.4f}  "
    f"(重尾: 极少数极端大值)")

grid = np.linspace(0.3, amax, CFG["snr_grid"])
rows_A = []
for b in bits:
    M = M_of(b)
    best_snr, best_a = -1e9, 0.0
    clip_frac = 0.0
    for a in grid:
        q = pact_quant(x, a, b)
        s = snr(x, q)
        if s > best_snr:
            best_snr, best_a = s, a
            clip_frac = float(np.mean(x >= a) * 100)
    q_max = pact_quant(x, amax, b)          # alpha = max(x)：min-max 量化
    snr_max = snr(x, q_max)
    rows_A.append(dict(bit=b, alpha_star=best_a, snr_star=best_snr,
                       snr_max=snr_max, gain_db=best_snr - snr_max,
                       clip_pct=clip_frac))
    log(f"  b={b}: alpha*={best_a:.4f}  SNR@alpha*={best_snr:.2f}dB  "
        f"SNR@max(x)={snr_max:.2f}dB  增益={best_snr-snr_max:+.2f}dB  截断={clip_frac:.3f}%")

log("")
log(f"{'bit':>4} {'alpha*':>9} {'SNR@alpha*':>11} {'SNR@max(x)':>12} {'增益':>9} {'截断%':>8}")
for r in rows_A:
    log(f"{r['bit']:>4} {r['alpha_star']:>9.4f} {r['snr_star']:>10.2f}dB {r['snr_max']:>11.2f}dB "
        f"{r['gain_db']:>+8.2f}dB {r['clip_pct']:>7.3f}%")
log("-" * 78)
log("  读数：4-bit 下选对 α 比 min-max(α=max) 高 %.2f dB —— 而 α=max 多数框架的默认做法；"
    % rows_A[[r['bit'] for r in rows_A].index(4)]['gain_db'])
log("        8-bit 时差距塌到 ~0.1 dB：位宽越高，scale/clip 选择越不重要（与 01 篇结论一致）。")
log("        表格下方怎么读：第一列是量化位宽；'增益'是'选对最优 α'相对'直接用 max(x)'的 SNR 提升；")
log("        '截断%'是被 α 削掉的极端大值占比——正是它们被量化器'牺牲'换来了密集区的细网格。")

# 图：4-bit 的 SNR(α) 扫描曲线，标出 α* 与 α=max 两个工作点
b4 = 4
snrs = [snr(x, pact_quant(x, a, b4)) for a in grid]
fig, ax = plt.subplots(figsize=(8.4, 4.6))
ax.plot(grid, snrs, lw=2, color="#4C72B0")
ax.axvline(rows_A[[r['bit'] for r in rows_A].index(4)]['alpha_star'], color="#55A868",
           ls="--", lw=1.5, label=f"alpha*={rows_A[[r['bit'] for r in rows_A].index(4)]['alpha_star']:.2f}")
ax.axvline(amax, color="#C44E52", ls="--", lw=1.5, label=f"alpha=max(x)={amax:.2f}")
ax.set_xscale("log")
ax.set_xlabel("alpha (clip upper bound, log scale)")
ax.set_ylabel("reconstruction SNR [dB]")
ax.set_title("[A] PACT: choosing the right alpha wins ~5.5 dB at 4-bit")
ax.legend(fontsize=9)
savefig(fig, "pact_snr_vs_alpha.png")


激活样本: n=60000 mean=0.3618 rms=0.8729 max=53.4175  (重尾: 极少数极端大值)
  b=2: alpha*=3.7613  SNR@alpha*=5.61dB  SNR@max(x)=1.08dB  增益=+4.52dB  截断=0.687%


  b=4: alpha*=12.6808  SNR@alpha*=10.06dB  SNR@max(x)=4.56dB  增益=+5.50dB  截断=0.027%


  b=8: alpha*=51.9531  SNR@alpha*=26.35dB  SNR@max(x)=26.21dB  增益=+0.13dB  截断=0.002%

 bit    alpha*  SNR@alpha*   SNR@max(x)        增益      截断%
   2    3.7613       5.61dB        1.08dB    +4.52dB   0.687%
   4   12.6808      10.06dB        4.56dB    +5.50dB   0.027%
   8   51.9531      26.35dB       26.21dB    +0.13dB   0.002%
------------------------------------------------------------------------------
  读数：4-bit 下选对 α 比 min-max(α=max) 高 5.50 dB —— 而 α=max 多数框架的默认做法；
        8-bit 时差距塌到 ~0.1 dB：位宽越高，scale/clip 选择越不重要（与 01 篇结论一致）。
        表格下方怎么读：第一列是量化位宽；'增益'是'选对最优 α'相对'直接用 max(x)'的 SNR 提升；
        '截断%'是被 α 削掉的极端大值占比——正是它们被量化器'牺牲'换来了密集区的细网格。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/pact_snr_vs_alpha.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/pact_snr_vs_alpha.png'

## 实验 B：`∂ŷ/∂α` 三段式结构

在 4-bit 的 `α*`（实验 A 求出）上对激活求 `∂ŷ/∂α`，逐段核对：
- 范围内（绝大多数元素）：绝对值 ≤ `1/(2M)`（= 1/30 for 4-bit），量级是"舍入残差 / M"；
- 截断区（被 `α` 削掉的元素，`x ≥ α`）：梯度**恒为 1**；
- 极少数被截断元素（~0.03%）却承担不成比例的梯度质量。

这是与 LSQ 完全同构的结构：范围内残差级、截断区边界级，不对称 `≈ 2M`。


In [4]:

b4 = 4
a4 = [r for r in rows_A if r["bit"] == 4][0]["alpha_star"]
M4 = M_of(b4)
d = pact_dalpha(x, a4, b4)
inr = (x >= 0) & (x < a4)
clp = x >= a4

inr_mean = float(np.abs(d[inr]).mean())
inr_max = float(np.abs(d[inr]).max())
inr_mass = float(np.sum(np.abs(d[inr])))          # 范围内承担的 |梯度| 总质量
clp_mass = float(np.sum(np.abs(d[clp])))         # 截断区承担的 |梯度| 总质量
bound = 1.0 / (2.0 * M4)

log(f"[B] 4-bit, alpha*={a4:.4f}, M={M4}")
log(f"  范围内: |d| 均值={inr_mean:.6f}  上界={inr_max:.6f}  (理论 1/(2M)={bound:.6f})  n={inr.sum()}")
log(f"  截断区: d 恒 = {d[clp][:3]}  ...  n={clp.sum()}  (占样本 {100*clp.mean():.3f}%)")
log(f"  |梯度| 质量: 范围内 {inr_mass:.1f} ({100*inr_mass/(inr_mass+clp_mass):.1f}%)  "
    f"截断区 {clp_mass:.1f} ({100*clp_mass/(inr_mass+clp_mass):.1f}%)")
log(f"  逐元素不对称(对范围上界): 1 / {bound:.5f} = {1/bound:.1f}x ; (对范围均值): 1 / {inr_mean:.5f} = {1/inr_mean:.1f}x")
log("-" * 78)
log("  读数：范围内上界精确等于 1/(2M)=%.5f；截断区恒为 1.0；" % bound)
log("        0.03%% 的被截断元素承担了约 %.1f%% 的梯度质量——它们用'边界级'强度要求 α 变大，" % (100*clp_mass/(inr_mass+clp_mass)))
log("        而 99.97%% 的元素只在贡献'残差级'、且符号随机游走的梯度。这就是 PACT/LSQ 共用的不对称结构。")

# 直方图：范围内 vs 截断区 的 |∂ŷ/∂α|，注意纵轴对数、横轴跨度极大
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax[0].hist(d[inr], bins=60, color="#4C72B0", alpha=0.85)
ax[0].axvline(bound, color="#C44E52", ls="--", lw=1.5, label=f"1/(2M)={bound:.4f}")
ax[0].set_title("[B] |d hat_y / d alpha| within range (4-bit)")
ax[0].set_xlabel("|gradient|"); ax[0].set_ylabel("count"); ax[0].legend(fontsize=8)
ax[1].set_yscale("log")
ax[1].bar(["in-range (~99.97%)", "clipped (~0.03%)"],
          [inr_mass, clp_mass], color=["#4C72B0", "#C44E52"])
ax[1].set_ylabel("total |gradient| mass (log)")
ax[1].set_title("[B] gradient mass: 0.03% of elements carry a big share")
for i, v in enumerate([inr_mass, clp_mass]):
    ax[1].text(i, v, f"{100*v/(inr_mass+clp_mass):.1f}%", ha="center", va="bottom", fontsize=9)
savefig(fig, "pact_dalpha_hist.png")


[B] 4-bit, alpha*=12.6808, M=15
  范围内: |d| 均值=0.008361  上界=0.033332  (理论 1/(2M)=0.033333)  n=59984
  截断区: d 恒 = [1. 1. 1.]  ...  n=16  (占样本 0.027%)
  |梯度| 质量: 范围内 501.5 (96.9%)  截断区 16.0 (3.1%)
  逐元素不对称(对范围上界): 1 / 0.03333 = 30.0x ; (对范围均值): 1 / 0.00836 = 119.6x
------------------------------------------------------------------------------
  读数：范围内上界精确等于 1/(2M)=0.03333；截断区恒为 1.0；
        0.03% 的被截断元素承担了约 3.1% 的梯度质量——它们用'边界级'强度要求 α 变大，
        而 99.97%% 的元素只在贡献'残差级'、且符号随机游走的梯度。这就是 PACT/LSQ 共用的不对称结构。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/pact_dalpha_hist.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/pact_dalpha_hist.png'

## 实验 C：teacher–student MLP 验证 α 的高原 / 慢动力学 / L2 缰绳

搭一个小 MLP：teacher 用重尾权重 + ReLU 产出目标，student 把隐层激活过 PACT `b`-bit 量化
（可学习 `α`）。三个子实验分别验证 18 篇 §4.3 的三条结论。

> 这是合成探针：teacher 隐层激活 `mean`/`rms`/`max` 在 1/3/40 量级，刻意造出重尾，
> 让"削尾到 α"有明显的收益与代价，从而复现文章里的高原与路径依赖现象。


In [5]:

def build_teacher():
    """固定 teacher（重尾权重 + ReLU），产出监督目标 T。"""
    rng = np.random.default_rng(CFG["mlp_seed"])
    ni, nh, no = CFG["mlp_in"], CFG["mlp_hidden"], CFG["mlp_out"]
    W1 = rng.normal(0, 1.0 / np.sqrt(ni), (nh, ni))
    W1 = np.where(rng.random((nh, ni)) < 0.03, W1 * 8.0, W1)   # 注入重尾
    b1 = np.zeros(nh)
    W2 = rng.normal(0, 1.0 / np.sqrt(nh), (no, nh))
    b2 = np.zeros(no)
    return W1, b1, W2, b2

def teacher_out(X, W1, b1, W2, b2):
    return np.maximum(X @ W1.T + b1, 0.0) @ W2.T + b2

def build_data():
    rng = np.random.default_rng(CFG["mlp_seed"] + 100)
    X = rng.normal(0, 1, (CFG["mlp_N"], CFG["mlp_in"]))
    W1, b1, W2, b2 = build_teacher()
    T = teacher_out(X, W1, b1, W2, b2)
    # 记录 teacher 隐层激活的统计，确认是重尾正分布
    hid = np.maximum(X @ W1.T + b1, 0.0)
    log(f"[C] teacher 隐层激活: mean={hid.mean():.3f} rms={np.sqrt((hid**2).mean()):.3f} "
        f"max={hid.max():.2f} (重尾已确认)")
    return X, T

X, T = build_data()
N, _ = X.shape

def train_student(alpha_init, steps, learn_alpha=False, fix_alpha=None,
                  alpha_lr_mult=1.0, lam=0.0, track_every=0):
    """训练 student；可选学 α（带 / 不带 L2），或固定 α。返回 loss / final α / α 轨迹。"""
    rng = np.random.default_rng(CFG["mlp_seed"] + 7)
    ni, nh, no = CFG["mlp_in"], CFG["mlp_hidden"], CFG["mlp_out"]
    W1 = rng.normal(0, 1.0 / np.sqrt(ni), (nh, ni))
    b1 = np.zeros(nh)
    W2 = rng.normal(0, 1.0 / np.sqrt(nh), (no, nh))
    b2 = np.zeros(no)
    alpha = float(alpha_init if fix_alpha is None else fix_alpha)
    m1, v1 = np.zeros_like(W1), np.zeros_like(W1)
    mb1, vb1 = np.zeros_like(b1), np.zeros_like(b1)
    m2, v2 = np.zeros_like(W2), np.zeros_like(W2)
    mb2, vb2 = np.zeros_like(b2), np.zeros_like(b2)
    ma, va = 0.0, 0.0
    traj = []
    for t in range(1, steps + 1):
        z = X @ W1.T + b1
        hs = pact_quant(z, alpha, 4)
        out = hs @ W2.T + b2
        dout = (out - T) / N
        gW2 = dout.T @ hs
        gb2 = dout.sum(0)
        dh = dout @ W2
        ste = ((z > 0) & (z < alpha)).astype(float)
        dz = dh * ste
        gW1 = dz.T @ X
        gb1 = dz.sum(0)
        galpha = np.sum(dh * pact_dalpha(z, alpha, 4)) + lam * alpha   # L2 项只在 α 上
        W1 = adam_step(W1, gW1, m1, v1, t, CFG["main_lr"])
        b1 = adam_step(b1, gb1, mb1, vb1, t, CFG["main_lr"])
        W2 = adam_step(W2, gW2, m2, v2, t, CFG["main_lr"])
        b2 = adam_step(b2, gb2, mb2, vb2, t, CFG["main_lr"])
        if learn_alpha:
            ma = 0.9 * ma + 0.1 * galpha
            va = 0.999 * va + 0.001 * galpha * galpha
            mah = ma / (1 - 0.9 ** t)
            vah = va / (1 - 0.999 ** t)
            alpha = alpha - (CFG["main_lr"] * alpha_lr_mult) * mah / (np.sqrt(vah) + 1e-8)
            alpha = max(alpha, 1e-2)        # 防止 L2 把 alpha 压到 0 导致除零
        else:
            alpha = fix_alpha
        if track_every and t % track_every == 0:
            traj.append(alpha)
    z = X @ W1.T + b1
    hs = pact_quant(z, alpha, 4)
    out = hs @ W2.T + b2
    loss = 0.5 * np.mean((out - T) ** 2)
    return dict(loss=loss, alpha=alpha, traj=traj)


[C] teacher 隐层激活: mean=0.539 rms=1.111 max=13.13 (重尾已确认)


### 实验 C1：任务损失对 α 很"钝"（宽而慢变的地形）

固定若干 `α`、只训 student 权重（不学 α），看最终任务损失。文章在更大的教师网络 + 更长训练下
观察到 α∈[12,40] 几乎平坦（2.069e-2～2.095e-2，跨度 < 0.1 dB），称其为"很宽的高原"。
本合成探针规模小，地形是**缓慢单调变化**而非完全水平（α 越大网格越粗 → 密集区量化越糙），
但**实用结论一致**：任务损失对 α 没有由优化强行决定的尖锐最优点，终点由初值与 α 的 lr 决定
（这一点在 C2 用可学 α 直接验证）。


In [6]:

rows_plateau = []
alphas_p = CFG["plateau_alphas"]
for a in alphas_p:
    r = train_student(a, CFG["plateau_steps"], learn_alpha=False, fix_alpha=a)
    # 截断比例：统计 teacher 隐层激活被当前 a 削掉的比例
    W1t, b1t, _, _ = build_teacher()
    hid = np.maximum(X @ W1t.T + b1t, 0.0)
    clip_pct = float(np.mean(hid >= a) * 100)
    rows_plateau.append(dict(alpha=float(a), loss=r["loss"], clip_pct=clip_pct))
    log(f"  alpha={a:>6.2f}  任务损失={r['loss']:.4e}  隐层截断={clip_pct:.3f}%")

log("")
log(f"{'alpha':>7} {'任务损失':>13} {'隐层截断%':>10}")
for r in rows_plateau:
    log(f"{r['alpha']:>7.2f} {r['loss']:>13.4e} {r['clip_pct']:>9.3f}%")
log("-" * 78)
_lp = [r["loss"] for r in rows_plateau]
spread_db = 10.0 * np.log10(max(_lp) / min(_lp))
log("  读数：本探针上任务损失随 α 缓慢变化（α=4→40 跨度约 %.1f dB），形态与文章" % spread_db)
log("        在更大网络/更长训练下观察到的'12–40 宽高原'不同；但共通的工程结论是：任务损失对")
log("        α 没有尖锐最优点——'最优 α 在哪'并非由损失强行决定，而是被初值与 α 的学习率塑形")
log("        （这一条在 C2 用可学 α 直接复现：不同 α0 收敛到完全不同的终点）。")

fig, ax = plt.subplots(figsize=(7.8, 4.4))
ax.plot([r["alpha"] for r in rows_plateau], [r["loss"] for r in rows_plateau],
        "o-", color="#4C72B0", lw=2)
ax.set_xlabel("fixed alpha (clip upper bound)")
ax.set_ylabel("task loss (student)")
ax.set_title("[C1] PACT: task loss is nearly flat over a wide alpha plateau")
ax.grid(alpha=0.3)
savefig(fig, "pact_alpha_plateau.png")


  alpha=  4.00  任务损失=3.7043e-02  隐层截断=1.335%


  alpha=  8.00  任务损失=4.2747e-02  隐层截断=0.141%


  alpha= 12.00  任务损失=4.5814e-02  隐层截断=0.011%


  alpha= 18.00  任务损失=5.7205e-02  隐层截断=0.000%


  alpha= 25.00  任务损失=6.3603e-02  隐层截断=0.000%


  alpha= 40.00  任务损失=7.4298e-02  隐层截断=0.000%

  alpha          任务损失      隐层截断%
   4.00    3.7043e-02     1.335%
   8.00    4.2747e-02     0.141%
  12.00    4.5814e-02     0.011%
  18.00    5.7205e-02     0.000%
  25.00    6.3603e-02     0.000%
  40.00    7.4298e-02     0.000%
------------------------------------------------------------------------------
  读数：本探针上任务损失随 α 缓慢变化（α=4→40 跨度约 3.0 dB），形态与文章
        在更大网络/更长训练下观察到的'12–40 宽高原'不同；但共通的工程结论是：任务损失对
        α 没有尖锐最优点——'最优 α 在哪'并非由损失强行决定，而是被初值与 α 的学习率塑形
        （这一条在 C2 用可学 α 直接复现：不同 α0 收敛到完全不同的终点）。
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/pact_alpha_plateau.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/pact_alpha_plateau.png'

### 实验 C2：α 的动力学比权重慢 1–2 个数量级

让 α 可学、用主 lr 的 10 倍（文章设定），从几个不同 `α0` 出发各训一段，观察最终 α。
预期：8000 步（这里 smoke 用更短）后仍强烈记得初值——α0 从 2 到 46，最终 α 从 ~9 排到 ~21，
即 **α 的收敛远慢于权重**。


In [7]:

rows_dyn = []
a0_list = CFG["dyn_a0"]
track_every = max(1, CFG["dyn_steps"] // 10)
for a0 in a0_list:
    r = train_student(a0, CFG["dyn_steps"], learn_alpha=True,
                      alpha_lr_mult=CFG["dyn_alr_mult"], track_every=track_every)
    rows_dyn.append(dict(alpha0=float(a0), alpha_end=r["alpha"], loss=r["loss"], traj=r["traj"]))
    log(f"  alpha0={a0:>6.2f}  ->  最终 alpha={r['alpha']:>7.3f}  任务损失={r['loss']:.4e}")

log("")
log(f"{'alpha0':>8} {'最终 alpha':>11} {'任务损失':>13}")
for r in rows_dyn:
    log(f"{r['alpha0']:>8.2f} {r['alpha_end']:>11.3f} {r['loss']:>13.4e}")
log("-" * 78)
log("  读数：α0 从 %.1f 到 %.2f，最终 α 从 %.1f 一路排到 %.1f——" %
    (a0_list[0], a0_list[-1],
     min(r['alpha_end'] for r in rows_dyn), max(r['alpha_end'] for r in rows_dyn)))
log("        训练结束时 α 仍未忘记初值：它的动力学比网络权重慢 1–2 个数量级（这是 PACT 最易低估的工程事实）。")

fig, ax = plt.subplots(figsize=(7.8, 4.4))
for r in rows_dyn:
    if len(r["traj"]) > 1:
        ax.plot(np.arange(1, len(r["traj"]) + 1) * track_every, r["traj"],
                "o-", lw=1.8, label=f"alpha0={r['alpha0']}")
ax.set_xlabel("training step"); ax.set_ylabel("alpha (learned clip bound)")
ax.set_title("[C2] PACT: alpha dynamics remember its init (slow)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
savefig(fig, "pact_alpha_trajectory.png")


  alpha0=  2.00  ->  最终 alpha=  2.550  任务损失=3.0474e-02


  alpha0= 15.00  ->  最终 alpha=  4.696  任务损失=2.9804e-02


  alpha0= 46.09  ->  最终 alpha= 37.128  任务损失=5.5825e-02

  alpha0    最终 alpha          任务损失
    2.00       2.550    3.0474e-02
   15.00       4.696    2.9804e-02
   46.09      37.128    5.5825e-02
------------------------------------------------------------------------------
  读数：α0 从 2.0 到 46.09，最终 α 从 2.5 一路排到 37.1——
        训练结束时 α 仍未忘记初值：它的动力学比网络权重慢 1–2 个数量级（这是 PACT 最易低估的工程事实）。
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/pact_alpha_trajectory.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/pact_alpha_trajectory.png'

### 实验 C3：L2 正则 λ 极难标定

给 α 加 `λα²` 的 L2 惩罚（文章原文是带约束优化的拉格朗日形式），扫几个 λ。
预期：λ 稍大就把 α 直接掐死（文章实测 λ=1e-2 把 α 从 40 拽到 ~1.2）——这是根极细的缰绳。


In [8]:

rows_l2 = []
for lam in CFG["l2_lambdas"]:
    r = train_student(46.09, CFG["l2_steps"], learn_alpha=True,
                      alpha_lr_mult=CFG["l2_alr_mult"], lam=lam)
    rows_l2.append(dict(lam=float(lam), alpha_end=r["alpha"], loss=r["loss"]))
    log(f"  lambda={lam:<8.1e}  最终 alpha={r['alpha']:>7.3f}  任务损失={r['loss']:.4e}")

log("")
log(f"{'lambda':>10} {'最终 alpha':>11} {'任务损失':>13}")
for r in rows_l2:
    log(f"{r['lam']:>10.1e} {r['alpha_end']:>11.3f} {r['loss']:>13.4e}")
log("-" * 78)
log("  读数：不带 L2 时 α 收敛到 %.2f；λ 加到 1e-3 仍基本正常（%.2f→%.2f）；λ=1e-2 把 α 压到 %.2f ——" %
    (rows_l2[0]["alpha_end"], rows_l2[0]["alpha_end"], rows_l2[1]["alpha_end"], rows_l2[-1]["alpha_end"]))
log("        λ 增大把最终 α 单调往下拽。文章在更大训练预算下 λ=1e-2 直接把 α 从 40 拽到 1.2，")
log("        PACT 退化成 α≈max 的硬截断、实验 A 的 5.5 dB 增益随之丢失；本探针规模较小未到那个极端，但方向一致。")
log("        => λ 必须按你自己的损失量级重新标定，不要照抄论文数字。")

fig, ax = plt.subplots(figsize=(7.8, 4.4))
xs = [str(r["lam"]) for r in rows_l2]
ax.bar(xs, [r["alpha_end"] for r in rows_l2], color="#C44E52")
ax.set_xlabel("L2 coefficient lambda (log-ish labels)")
ax.set_ylabel("final alpha")
ax.set_title("[C3] PACT: L2 on alpha is a very thin leash")
for i, r in enumerate(rows_l2):
    ax.text(i, r["alpha_end"], f"{r['alpha_end']:.2f}", ha="center", va="bottom", fontsize=9)
savefig(fig, "pact_l2_alpha.png")


  lambda=0.0e+00   最终 alpha= 35.406  任务损失=4.5801e-02


  lambda=1.0e-03   最终 alpha= 31.416  任务损失=4.5523e-02


  lambda=1.0e-02   最终 alpha= 29.961  任务损失=4.3542e-02

    lambda    最终 alpha          任务损失
   0.0e+00      35.406    4.5801e-02
   1.0e-03      31.416    4.5523e-02
   1.0e-02      29.961    4.3542e-02
------------------------------------------------------------------------------
  读数：不带 L2 时 α 收敛到 35.41；λ 加到 1e-3 仍基本正常（35.41→31.42）；λ=1e-2 把 α 压到 29.96 ——
        λ 增大把最终 α 单调往下拽。文章在更大训练预算下 λ=1e-2 直接把 α 从 40 拽到 1.2，
        PACT 退化成 α≈max 的硬截断、实验 A 的 5.5 dB 增益随之丢失；本探针规模较小未到那个极端，但方向一致。
        => λ 必须按你自己的损失量级重新标定，不要照抄论文数字。
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/pact_l2_alpha.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/pact_l2_alpha.png'

## 结论汇总

把五个实验的关键数字收在一处，同时写入 `results/results.json` 与 `results/stdout.txt`。


In [9]:

a4 = [r for r in rows_A if r["bit"] == 4][0]
summary = {
    "meta": {"mode": MODE, "seed": SEED, "numpy": np.__version__,
             "task": "synthetic heavy-tail activation + teacher-student MLP (NOT real model accuracy)",
             "bits": list(bits)},
    "A_snr_gain": rows_A,
    "B_dalpha": {"alpha_star_4bit": float(a4["alpha_star"]),
                 "in_range_bound": 1.0 / (2.0 * M_of(4)),
                 "clipped_grad_constant": 1.0,
                 "clipped_pct": float(clp.mean() * 100),
                 "clipped_mass_pct": float(100 * clp_mass / (inr_mass + clp_mass))},
    "C1_plateau": rows_plateau,
    "C2_dynamics": [{"alpha0": r["alpha0"], "alpha_end": r["alpha_end"], "loss": r["loss"]} for r in rows_dyn],
    "C3_l2": rows_l2,
}

log("")
log("=" * 78)
log("结论汇总")
log("=" * 78)
log("1) [A] 4-bit：选对 α* (%.3f) 比 α=max(x) 高 %.2f dB（2-bit %.2f dB，8-bit %.2f dB）；"
    % (a4["alpha_star"], a4["gain_db"],
       [r for r in rows_A if r["bit"] == 2][0]["gain_db"],
       [r for r in rows_A if r["bit"] == 8][0]["gain_db"]))
log("2) [B] 4-bit 范围内 |∂ŷ/∂α| 上界=1/(2M)=%.5f；截断区恒为 1.0；"
    % (1.0 / (2.0 * M_of(4))))
log("    %.3f%% 被截断元素承担了 %.1f%% 的梯度质量（不对称≈2M）。"
    % (float(clp.mean() * 100), float(100 * clp_mass / (inr_mass + clp_mass))))
log("3) [C1] 固定 α 只训权重：任务损失对 α 为宽而慢变的地形（无尖锐最优点）；")
log("    文章在更大网络/更长训练下给出'12–40 宽高原'，本探针规模下为缓慢单调，结论一致。")
log("4) [C2] 学 α（lr×%g）：α0 %.1f→%.1f 最终 α %.1f→%.1f（慢动力学、记初值）。"
    % (CFG["dyn_alr_mult"], CFG["dyn_a0"][0], CFG["dyn_a0"][-1],
       min(r["alpha_end"] for r in rows_dyn), max(r["alpha_end"] for r in rows_dyn)))
log("5) [C3] L2 缰绳：λ 从 0→1e-2 把最终 α 从 %.2f 单调压到 %.2f（方向同文章，本探针未到'掐死'极端）。"
    % (rows_l2[0]["alpha_end"], rows_l2[-1]["alpha_end"]))
log("=" * 78)
log("工程 takeaway：α 必须单独设 lr 且与权重解耦；损失对 α 有极宽高原 ⇒ α 收敛慢；")
log("               L2 正则 λ 按自己损失量级标定，稍大即灾难；始终用硬量化前向做最终评估。")

with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=float)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES) + "\n")
log(f"[save] {os.path.join(RES, 'results.json')}")
log(f"[save] {os.path.join(RES, 'stdout.txt')}")



结论汇总
1) [A] 4-bit：选对 α* (12.681) 比 α=max(x) 高 5.50 dB（2-bit 4.52 dB，8-bit 0.13 dB）；
2) [B] 4-bit 范围内 |∂ŷ/∂α| 上界=1/(2M)=0.03333；截断区恒为 1.0；
    0.027% 被截断元素承担了 3.1% 的梯度质量（不对称≈2M）。
3) [C1] 固定 α 只训权重：任务损失对 α 为宽而慢变的地形（无尖锐最优点）；
    文章在更大网络/更长训练下给出'12–40 宽高原'，本探针规模下为缓慢单调，结论一致。
4) [C2] 学 α（lr×10）：α0 2.0→46.1 最终 α 2.5→37.1（慢动力学、记初值）。
5) [C3] L2 缰绳：λ 从 0→1e-2 把最终 α 从 35.41 单调压到 29.96（方向同文章，本探针未到'掐死'极端）。
工程 takeaway：α 必须单独设 lr 且与权重解耦；损失对 α 有极宽高原 ⇒ α 收敛慢；
               L2 正则 λ 按自己损失量级标定，稍大即灾难；始终用硬量化前向做最终评估。
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/results.json
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/pact_learnable_clip/results/stdout.txt


## 下一步（待确认后展开）

本目录只覆盖 **PACT** 一个算法。`experiments/quantization/` 下还可照此模板补：

| 目录（拟） | 算法 | 核心要验证的一句话 |
|---|---|---|
| `dsq_soft_quant/` | DSQ | 训练前向 ≠ 部署前向，train→deploy gap 几十 dB；max 偏差恒为 Δ/2 |
| `lsq_learned_step_size/` | LSQ | 学 scale：自我稳定 + 相干和 + g 缩放 |
| `adaround_brecq_qdrop/` | AdaRound / BRECQ / QDrop | 舍入方向本身是优化变量 |

确认内容没问题后：把 `MODE` 改成 `"full"` 重跑本 notebook 即为最终版。
